# DenseNet-121 Best Pipeline — ChestMNIST

Puedes ejecutar el experimento de dos formas:

**A) Script desatendido** (reproducible, sin Jupyter):


**B) Interactivo** (esta notebook): permite inspeccionar resultados entre pasos.

Artefactos generados:
- 
- , 
-  (todas las gráficas)


## 0. Setup

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath("")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from medmnist import INFO
from src.config.config import DATA_FLAG, DEVICE

info = INFO[DATA_FLAG]
N_CLASSES = len(info["label"])
print(f"Dispositivo: {DEVICE}")
print(f"Clases: {N_CLASSES}")

## 1. Configuración del experimento

Los parámetros de entrenamiento viven en .
Instanciarlos aquí hace el experimento explícito y reproducible.

In [ ]:
from src.entity.schemas import OptunaConfig, TrainingConfig

optuna_cfg = OptunaConfig(
    n_trials=25,
    n_epochs=8,
    patience=3,
)

train_cfg = TrainingConfig(
    n_epochs=30,
    patience=5,
)

## 2. Búsqueda de hiperparámetros con Optuna

| Hiperparámetro | Rango |
|---|---|
|  | [1e-5, 5e-4] (log) |
|  | {32, 64} |
|  | [2, 6] |
|  | [0, 2] |
|  | [0.0, 0.1] |
|  | [0, 1e-3] |
|  | {cosine, step, none} |
|  | [10, 20] |
|  | [0.05, 0.15] |

Gráfica guardada en .

In [ ]:
from src.pipelines.hparam_search import run_hparam_search

best_params = run_hparam_search(optuna_cfg)

## 3. Entrenamiento final

Curvas guardadas en .
Modelo guardado en .

In [ ]:
from src.pipelines.train_pipeline import run_training

model, history, cfg = run_training(best_params, train_cfg)

## 4. Calibración de umbrales y evaluación en test

Gráficas guardadas en  y .
Métricas guardadas en .

In [ ]:
from src.pipelines.eval_pipeline import run_evaluation

(
    final_metrics, optimal_thresholds,
    y_score_test, y_true_test, y_pred_test
) = run_evaluation(model, cfg)

## 5. Explicabilidad con Grad-CAM++

7 paneles guardados en .

In [ ]:
from src.pipelines.gradcam_pipeline import run_gradcam

run_gradcam(model, optimal_thresholds)